# TIDMAD: change one band, save it, and run your saved experiment

## Start here: Run All → three search iterations → your progress plot

**Run All performs paid LLM calls and real GPU training through your project's saved script.** Before starting, initialize this external project, prepare the data, and launch Jupyter from a terminal with the required API keys exported. Use the exp checkout's kernel. Do not paste keys into cells.

The first section below is a complete short search demo: **save its configuration → review the saved files → invoke its script → plot score versus iteration**. It uses three search iterations, not three training batches. LLM work can dominate runtime; this is a small search, not a promised seconds-long example. A finite or valid score is not guaranteed.

Set `RUN_QUICK_DEMO=False` to browse without starting a new run. Re-running the same completed demo reuses its records and redraws the figure without another API call. To run changed settings, choose a new `DEMO_NAME`; previous results are retained.

After the plot, the advanced sections teach individual parameter edits and task splits. Their save switches remain opt-in. **The quick demo ends at the validation progress plot; it never automatically chooses a model for an independent final test.**


A model takes a noisy detector waveform and predicts its clean injected waveform. We use **band `0-3`**, so you can learn the configuration without setting up four separate band searches.

## Read this first: two stages, with a model-selection decision between them

**Stage A is the model search:** train candidate models and use **workflow validation** to decide what to improve. **Stage B is a final test:** after the search stops, evaluate one selected model on data that did not guide that search. These stages answer different questions.

```text
PREPARE — notebook, sections 1–5
  choose data → edit task/experiment → save files
       ↓
SEARCH — terminal command from section 6
  train on files [0,1] → validate on file [2] → agent adjusts the model/settings
  Trial AND Formal belong to this search; neither is the final test
       ↓
REVIEW — notebook, section 7
  inspect validation/Health records → stop search → choose one successful model
       ↓  only if using file-holdout and a successful model exists
FINAL TEST — separate terminal commands from section 8
  --seal: record the chosen model/config hashes; NO training or scoring
  --evaluate: infer and score on held-out file [3]; NO training or LLM calls
       ↓
READ OUTPUTS — section 9
  search records and final-test result remain separate
```

The file numbers above are the example defaults; section 4 lets you change them. Validation is visible to the agent, so it cannot also serve as an untouched final test. **Formal means a training/evaluation stage inside the search, not final testing.**

| Section | What you do | What you should have when finished |
|---|---|---|
| 1. Files and responsibilities | Locate your external project, task directory and experiment JSON | Know which file owns each choice |
| 2. Data | Reuse shared data OR download on your own machine | A prepared `data/band-0-3/` directory |
| 3. Experiment controls | Choose iterations, fractions, time and VRAM | Edited values in notebook memory |
| 4. Task split | Choose training/validation/final-test file groups | A new split definition in notebook memory |
| 5. Save | Write the task YAML, experiment JSON and matching scripts | Files on disk that a launcher can read |
| 6. Check and launch search | Review the saved-file checklist; run its preview, then its launch command | Search records and trained checkpoints under `runs/` |
| 7. Inspect the search | Read validation scores, status and Health; finish model selection | One eligible model, or a reason to stop/revise the search |
| 8. Final test | Save the selected candidate, seal it, then evaluate once | A separate held-out result under `final-test/` |
| 9. Locate outputs | Open the appropriate search or test result | Know which result answers which question |

**When may I stop?** If you only want to learn configuration and launching, sections 1–7 are enough. To demonstrate the independent final test for `file-holdout`, continue to section 8 after model selection. The initial `paper-pool` example has no independent final-test partition and cannot use this final-test entrypoint. If all search attempts fail, there is no eligible model: do not proceed to section 8.

The notebook explains, edits and inspects. The quick-demo cell invokes a saved terminal script; the advanced route lets you invoke such scripts yourself. Keep both source repositories unchanged.

This notebook must be the copy in your external project. If you have not created one, run the following from the installed exp checkout first, then open the copied notebook with that checkout's Jupyter kernel:

```bash
export TUTORIAL_HOME=/absolute/path/to/my-tidmad-study
.venv/bin/python -B -m tutorials.paper.tidmad.project --project "$TUTORIAL_HOME" --infra-checkout /absolute/path/to/SIDERIUS-tutorial
.venv/bin/jupyter lab --ServerApp.root_dir="$TUTORIAL_HOME"
```

`TUTORIAL_HOME` is your study folder, not the exp repo and not a single run's output folder.

**Where do I actually start the run?** Save your changes, then go to **section 6: Before you run**. It gives the file checklist, actual saved parameters, and exact terminal command. Section 8 is final testing after search, not the training launch.
Before launching, both the notebook and saved script check the environment. An error lists missing files or key names and how to fix them. Keys must be exported before starting Jupyter; a kernel restart alone cannot import variables added later to a different terminal. The native launcher then verifies source revisions, data integrity and CUDA. Key presence does not guarantee provider access or available credit.


In [ ]:
import json
import os
import shlex
import sys
from pathlib import Path

project_env = os.environ.get("TUTORIAL_HOME")
if not project_env:
    raise RuntimeError(
        "Missing TUTORIAL_HOME. In your terminal, export TUTORIAL_HOME=/absolute/path/to/your/tutorial-project, "
        "then restart Jupyter from that terminal. Use the external project created in the README setup step."
    )
PROJECT = Path(project_env).expanduser().resolve()
if not (PROJECT / "project.json").is_file():
    raise RuntimeError(
        f"Missing {PROJECT / 'project.json'}. Point TUTORIAL_HOME at your initialized external project, "
        "or complete the README's create-project command first. Do not use the source repo as your project."
    )
binding = json.loads((PROJECT / "project.json").read_text())
EXP, INFRA = Path(binding["exp_checkout"]), Path(binding["infra_checkout"])
if Path(sys.prefix).resolve() != (EXP / ".venv").resolve():
    raise RuntimeError(
        f"Wrong notebook kernel: {sys.executable}. Select the kernel installed from {EXP / '.venv/bin/python'}. "
        f"If missing, run: cd {shlex.quote(str(EXP))} && uv sync --group dev --group tutorial --frozen; "
        "then register/select that environment's Jupyter kernel as shown in the README."
    )
import importlib.util

missing_modules = [name for name in ("pydantic", "yaml", "numpy", "h5py", "torch", "workflows", "matplotlib")
                   if importlib.util.find_spec(name) is None]
if missing_modules:
    raise RuntimeError(
        f"Missing Python modules: {', '.join(missing_modules)}. "
        f"Run: cd {shlex.quote(str(EXP))} && uv sync --group dev --group tutorial --frozen; then restart this kernel."
    )
sys.dont_write_bytecode = True
from tasks.tidmad.runtime.file_split import FileSplit, sample_population
from tutorials.paper.tidmad.project import write_file_split_task, write_launcher
from tutorials.paper.tidmad.runner import TidmadExperiment

base_path = PROJECT / "experiments/tidmad-experiment.json"
base = TidmadExperiment.model_validate_json(base_path.read_text())
print("YOUR PROJECT:", PROJECT)
print("Initial saved experiment:", base_path)
print("Initial task entry:", base.composition)
print("Initial launcher:", PROJECT / "scripts/run-tidmad.sh")

## Quick demo A — save and check the actual inputs

The next cell creates a new experiment JSON and matching shell script **inside your project**, then shows their paths, saved parameter table, data checks and exact launch command. It never overwrites a different experiment under the same name. If you see **STOP**, fix the stated prerequisite before running the search cell.

You can change the numbers below before the first run. `iterations=3` is the outer search count; each iteration can contain Trial/Formal attempts and retries. Five epochs is a ceiling, and the native per-attempt budget may stop earlier. `gpu` and `data_dir` inherit your project's initial experiment; change those saved initial bindings before creating a new demo if necessary.

This quick demo copies a separate task with train files `[0,1]`, workflow validation `[2]`, and reserved final-test `[3]`. Trial uses 1% and Formal uses 2% within those groups: respectively 2/4 original PSD segments per assigned file. Training receives up to 2/5 minutes per attempt, with a five-epoch ceiling; these are allowances, not mandatory durations. Literature review remains enabled; this contributes to runtime. File 3 is not evaluated automatically.


If this demo already completed, its workspace is expected to exist: the next cell reuses the saved run rather than relaunching it. A suppressed new-launch command is not a failure of result reuse.

The fixed TIDMAD workflow separately uses 10% of its validation file for per-epoch validation loss: 20 segments from file 2. The `*_val_fraction` fields above control inference/scoring, not that separate training-validation workload. The time allowance must cover both training and that validation pass.


In [ ]:
from IPython.display import Markdown, display

from tutorials.paper.launch_review import launch_review
from tutorials.paper.quick_demo import prepare_demo, run_demo

DEMO_NAME = "quick-demo-001"  # Use a NEW name when changing saved settings.
RUN_QUICK_DEMO = True  # Run All will invoke the script: paid APIs + GPU work.
QUICK_SETTINGS = {
    "iterations": 3,
    "epochs": 5,
    "trial_minutes": 2.0,
    "formal_minutes": 5.0,
    "vram_gib": 8.0,
    "trial_train_fraction": 0.01,
    "trial_val_fraction": 0.01,
    "formal_train_fraction": 0.02,
    "formal_val_fraction": 0.02,
}
demo = prepare_demo(PROJECT, "tidmad", name=DEMO_NAME, settings=QUICK_SETTINGS)
display(Markdown(launch_review(demo.experiment, demo.script, task="tidmad")))

## Quick demo B — run the saved script and wait

`run_demo` invokes **the exact shell script displayed above with `--launch`**. The script remains responsible for key/data/GPU checks and workflow execution; the notebook does not contain training code. Keep the kernel running. This cell prints elapsed time and the external console-log path while the search runs.

If a model fails, the real failure records are retained and plotted. If launch fails before producing records, inspect the log and fix the prerequisite rather than treating an empty graph as success. Kernel interruption stops the child process group; an interrupted run is not silently restarted.


In [ ]:
if RUN_QUICK_DEMO:
    demo_result = run_demo(demo)
    print(demo_result)
else:
    print("Execution skipped. The next cell can plot an existing run.")

## Quick demo C — see what the search actually did

**Read this as a validation learning curve, not a final-test score.** The x-axis is the outer search iteration; points are recorded **Formal** attempts, never substituted Trial scores.

- **Filled circle:** successful attempt with measured Health PASS.
- **Hollow circle:** Health FAIL or a failed attempt. A raw metric may exist even when the accepted score is null; it is shown as invalid, not promoted to a valid result.
- **Dashed line:** Formal results. **Solid line:** current best. **Star:** a new best with Health PASS.
- As in the paper, negative scores appear as triangles at the axis boundary; exact values remain in the CSV. Missing scores and unknown Health evidence remain in the CSV without an extra marker or panel.

As in the paper's three-task progress figure, the line tracks the best **recorded** Formal score, including hollow invalid points. A high line therefore does not prove a healthy model. This tutorial keeps the paper’s plot style and uses iteration rather than elapsed time. Online Health is not the paper's separate retrospective behavioral review.

The cell saves a PNG, SVG and a CSV containing each point's iteration, attempt ID, status and source record. For another run, change **only `PLOT_WORKSPACE`** to its run directory and execute this plotting cell; this does not launch a search. The raw records remain the authority.


In [ ]:
from tutorials.paper.progress import plot_progress

PLOT_WORKSPACE = demo.workspace  # Or Path("/your/project/runs/another_run")
PLOT_OUTPUT = PROJECT / "plots" / PLOT_WORKSPACE.name
if PLOT_WORKSPACE.is_dir():
    progress_figure = plot_progress(
        PLOT_WORKSPACE,
        PLOT_OUTPUT,
        title="TIDMAD quick demo",
        expected_iterations=QUICK_SETTINGS["iterations"]
        if PLOT_WORKSPACE == demo.workspace
        else None,
    )
    display(progress_figure)
    import matplotlib.pyplot as plt

    plt.close(progress_figure)
    print("PNG:", PLOT_OUTPUT / "score-versus-iteration.png")
    print("SVG:", PLOT_OUTPUT / "score-versus-iteration.svg")
    print("Point records:", PLOT_OUTPUT / "score-versus-iteration.csv")
else:
    print(
        "No run workspace yet. Run Quick demo B or select an existing PLOT_WORKSPACE."
    )

## Advanced walkthrough — edit individual choices deliberately

The quick demo above is complete once you have inspected its plot and records. The sections below explain each file and control in more detail. Their save cells create separate examples; they do not overwrite or rerun the quick demo.

## 1. What are a task and an experiment on disk?

A **task package is a directory of related files**, not a single notebook. Its composition YAML is the entry file: it connects the data adapter, waveform contract, metric and Health rules. An **experiment is one JSON file**: it points to that task entry and sets budgets, model routing, data and output paths. A **launcher is one shell script**: its `EXPERIMENT` variable names the JSON it will read.

```text
YOUR_PROJECT/
├── notebooks/02_tidmad_tutorial.ipynb          your editable notebook
├── tasks/tidmad/                              the complete task package
│   ├── compositions/continuous_regression_frozen_pool.yaml  original paper pool
│   ├── compositions/file_holdout.yaml         created by the save step below
│   ├── declared/task_config_regression.yaml   waveform input/output contract
│   ├── resolved/dataset_profile.json          file names, channels, segment sizes
│   ├── resolved/metric_spec.json              what score means
│   ├── runtime/file_split.py                  implements your file membership
│   └── reference_data/                        fixed scoring references
├── experiments/tidmad-experiment.json         initial paper-pool demo
├── experiments/tidmad-file-split.json         created by the save step below
├── llm/agents.json                            model/provider choices, NO keys
├── advice/README.txt                          explains why advice is inactive
├── scripts/run-tidmad-file-split.sh           created by the save step below
├── scripts/test-tidmad-file-split.sh          separate final-test entrypoint
├── data/band-0-3/                             run inputs, NOT result files
├── runs/tidmad_file_split_001/                created only by a real launch
└── final-test/                               selected model and test results
```

The first save step creates the `file_holdout.yaml`, experiment and scripts together. Before that, those paths in this diagram are **planned filenames**, not files that already exist. Editing a notebook variable changes only kernel memory until you explicitly save it.

## 2. Choose ONE data entrance

**A — On the shared RTX 5090 machine:** use the existing `/home/klz/Data/TIDMAD`. The command below verifies the eight source files and creates eight small symbolic links in your project's `data/band-0-3/`. It does not download, copy, rename or alter the large HDF5 files. The local `segment_anchors.json` was supplied by project initialization. The links remain dependent on the shared source paths staying available.

**B — On your own machine:** obtain the official [TIDMAD downloader](https://github.com/jessicafry/TIDMAD) outside the source checkouts and download only four training and four validation files into your project's `data/band-0-3/`. Do not download the science dataset for this tutorial. One band still represents about 32 GB of uncompressed channel samples.

The next cell **prints** the command for your selected entrance; run that command once in a terminal. If your data is already prepared, skip preparation and inspect the paths instead. Both entrances produce the same `data_dir`, so the experiment and launch script do not need different implementations.

In [ ]:
DATA_ENTRY = "existing"  # "existing" on the shared machine; "download" elsewhere
if DATA_ENTRY == "existing":
    command = [
        str(EXP / ".venv/bin/python"),
        "-B",
        "-m",
        "tutorials.paper.data_entry",
        "--task",
        "tidmad",
        "--project",
        str(PROJECT),
        "--source",
        "/home/klz/Data/TIDMAD",
    ]
elif DATA_ENTRY == "download":
    command = [
        sys.executable,
        "/absolute/path/to/TIDMAD/download_data.py",
        "--output_dir",
        str(base.data_dir),
        "--train_files",
        "4",
        "--validation_files",
        "4",
        "--science_files",
        "0",
    ]
else:
    raise ValueError("Choose existing or download")
print("Run once from the exp checkout:", shlex.join(command))
print("Experiment data_dir:", base.data_dir)
for path in sorted(base.data_dir.glob("*")):
    print(path.name, "→", path.resolve() if path.is_symlink() else "local file")

## 3. Change the experiment: iterations, fractions and budgets

An **iteration** is another search cycle. An **epoch** is a pass through an attempt's selected training data. **Trial** tries a smaller workload and settings; **Formal** is the larger training/evaluation stage. Failures and retries mean these are not exact counts of successful runs.

| Change | JSON field | Meaning |
|---|---|---|
| Search longer | `iterations` | Search iteration ceiling |
| Train longer per attempt | `epochs` | Trial/Formal epoch ceiling |
| Trial training amount | `trial_train_fraction` | Portion of eligible training segments |
| Trial evaluation amount | `trial_val_fraction` | Portion of eligible validation segments |
| Formal training amount | `formal_train_fraction` | Portion of eligible training segments; paper-pool exception below |
| Formal evaluation amount | `formal_val_fraction` | Portion of eligible validation segments |
| Time allowance | `trial_minutes`, `formal_minutes` | Native per-stage training budgets, not a total script deadline |
| Memory allowance | `trial_vram_gib`, `formal_vram_gib` | GiB; leave GPU headroom |

Use fractions such as `.25`, not `25`. Changing a fraction samples fewer segments **inside an existing split**; it does not move a file from train to validation.

The original **paper-pool** task fixes 20 training PSD segments out of 200 per file. Trial `.5` means **10/20**. Formal `.1` declares the original **20/200** parent pool and uses all 20; it does not take another 10%. This entrypoint refuses other Formal training fractions for paper-pool. Evaluation is independent, over each original validation file's 200 segments. To freely change Formal fractions, use the new task in section 4.

In [ ]:
def variant(name, **changes):
    values = base.model_dump(mode="json")
    values.update(changes, run_name=name, workspace=str(PROJECT / "runs" / name))
    return TidmadExperiment.model_validate(values)


longer = variant("tidmad_longer_001", iterations=3, epochs=10)
less_trial = variant(
    "tidmad_less_trial_001", trial_train_fraction=0.25, trial_val_fraction=0.1
)
budgets = variant(
    "tidmad_budget_001",
    trial_minutes=3,
    formal_minutes=10,
    trial_vram_gib=6,
    formal_vram_gib=8,
)
for name, settings in [
    ("longer", longer),
    ("less-trial", less_trial),
    ("budgets", budgets),
]:
    print(name, "will save to", PROJECT / "experiments" / f"tidmad-{name}.json")
    print("  output after launch:", settings.workspace)

## 4. Change the task: hold out different file indices

For this simpler example, file indices represent different frequency ranges. Change the three lists below. The example uses **training files 0 and 1**, **validation file 2**, and **final-test file 3**:

| Role | Physical data read | Feedback |
|---|---|---|
| Training | `abra_training_0000.h5`, `abra_training_0001.h5` | Used to fit model weights |
| Workflow validation | `abra_validation_0002.h5` | Visible to the agent; can affect training/model selection |
| Final test | `abra_validation_0003.h5` | Separate inference after you select and fix one model |

Other family/index combinations are excluded from these scopes. This is **file-index / frequency-range holdout**, not a per-frequency label classification. It requires no FFT catalog or signal-strength threshold. Every original segment within each assigned file remains eligible, including weak-signal segments. File boundaries do not prove that exact frequencies or spectral content can never overlap.

Each list must be nonempty, the lists must not overlap, and together they must contain `0,1,2,3`. For example, changing validation to `[1]` also requires removing `1` from training and assigning `2` elsewhere. Do not rename or reorder raw files; scoring uses original file/segment identities.

The saved membership belongs to `tasks/tidmad/compositions/file_holdout.yaml`, under `task_data_path.config.split`. This new task uses ordinary scopes instead of the frozen paper pool. It changes the scientific protocol, so its scores are not a direct reproduction of the frozen paper experiment.

In [ ]:
TRAIN_FILES = [0, 1]
VALIDATION_FILES = [2]
FINAL_TEST_FILES = [3]
file_split = FileSplit(
    train_files=tuple(TRAIN_FILES),
    validation_files=tuple(VALIDATION_FILES),
    test_files=tuple(FINAL_TEST_FILES),
    seed=42,
)
print("The split that will be saved into your task YAML:")
print(file_split.model_dump_json(indent=2))
for role in ("train", "validation", "test"):
    print(
        role, {index: len(rows) for index, rows in file_split.population(role).items()}
    )

### Combine the new split with Trial/Formal proportions

Each assigned file contains 200 eligible PSD segments. With training files `[0,1]`, Trial `.25` selects 50 from each file (**100 total**); Formal `.5` selects 100 from each (**200 total**). With validation file `[2]`, Trial `.5` selects **100** and Formal `1.0` selects **200**. The final test remains all 200 segments of file 3 and is never part of these workflow scopes.

The custom adapter uses `ceil(fraction × eligible segments)` per file, at least one. A one-second PSD segment contains 10M samples, or 250 model windows of 40,000 samples. Smaller fractions can still fail the existing runtime calibration checks. Raise an inadequate allowance instead of disabling checks.

In [ ]:
for label, role, portion in [
    ("Trial training", "train", 0.25),
    ("Formal training", "train", 0.5),
    ("Trial validation", "validation", 0.5),
    ("Formal validation", "validation", 1.0),
]:
    counts = {
        i: len(rows)
        for i, rows in sample_population(
            file_split.population(role), portion, 42
        ).items()
    }
    print(label, counts, "total:", sum(counts.values()))

## 5. Save: this is where notebook edits become files

Set `SAVE_EXAMPLES=True` in the next cell and run it **once**. It creates the new task entry plus four experiment/script pairs. No training starts. Files are not silently overwritten; for a second study choose new filenames/run names (or edit the saved JSON directly in Jupyter).

After saving, changing `TRAIN_FILES` in an earlier cell alone does not update the YAML. Likewise, changing `budgets` in memory does not update a JSON already on disk. The launcher always reads the **saved** files. Save a new variant, or open and deliberately edit the saved file you intend to run.

The next section reads those files back so you can verify exactly what was saved.

In [ ]:
SAVE_EXAMPLES = False
saved = {}
if SAVE_EXAMPLES:
    task_entry = write_file_split_task(PROJECT / "tasks/tidmad", file_split)
    file_experiment = variant(
        "tidmad_file_split_001",
        protocol="file-holdout",
        composition=str(task_entry),
        trial_train_fraction=0.25,
        trial_val_fraction=0.5,
        formal_train_fraction=0.5,
        formal_val_fraction=1.0,
    )
    for name, settings in [
        ("longer", longer),
        ("less-trial", less_trial),
        ("budgets", budgets),
        ("file-split", file_experiment),
    ]:
        config = PROJECT / "experiments" / f"tidmad-{name}.json"
        script = PROJECT / "scripts" / f"run-tidmad-{name}.sh"
        with config.open("x") as f:
            f.write(settings.model_dump_json(indent=2))
        write_launcher(script, config, INFRA)
        saved[name] = config
    write_launcher(
        PROJECT / "scripts/test-tidmad-file-split.sh",
        saved["file-split"],
        INFRA,
        final=True,
    )
    print("Saved task:", task_entry)
for name in ("longer", "less-trial", "budgets", "file-split"):
    path = PROJECT / "experiments" / f"tidmad-{name}.json"
    print("SAVED" if path.exists() else "NOT SAVED", path)

## 6. Before you run: one experiment, one checklist, one launch command

For the new train/validation/test split, select **`file-split`**. It means exactly:

```text
scripts/run-tidmad-file-split.sh               ← the file bash executes
  └─ experiments/tidmad-file-split.json        ← fractions, budgets, paths
       ├─ tasks/tidmad/compositions/file_holdout.yaml ← the three file groups
       ├─ llm/agents.json                     ← provider/model routing, no keys
       ├─ data/band-0-3/                      ← existing raw data links or downloads
       └─ runs/tidmad_file_split_001/         ← NEW search results
```

The JSON's actual paths win if you edited them. The report below prints those actual paths; this diagram is only the example.

| Select `RUN_EXAMPLE` | Saved experiment in `experiments/` | Executed file in `scripts/` |
|---|---|---|
| `initial` | `tidmad-experiment.json` | `run-tidmad.sh` |
| `longer` | `tidmad-longer.json` | `run-tidmad-longer.sh` |
| `less-trial` | `tidmad-less-trial.json` | `run-tidmad-less-trial.sh` |
| `budgets` | `tidmad-budgets.json` | `run-tidmad-budgets.sh` |
| `file-split` | `tidmad-file-split.json` | `run-tidmad-file-split.sh` |

The initial `run-tidmad.sh` still selects the paper pool. It does not switch to the new file split automatically. **Do not use `test-tidmad-file-split.sh` to start training:** that is the separate post-selection final-test entrypoint in section 8.


**Do this once for the ONE experiment you intend to run:**

1. Save your edited experiment and its matching script using the earlier save cell (or use the initial experiment).
2. Select its name in the next cell. Run that cell to reload the actual files from disk.
3. In output **A**, open the listed files in JupyterLab's file browser and check their connections. In **B**, check every saved parameter. In **C**, check input files, a fresh output path and required key names.
4. If something is wrong, edit the named file, **save it**, and rerun this cell. Do not rely on values left in notebook memory.
5. In output **D**, copy the **preview line** into a terminal. After checking preview and exporting keys there, copy the **launch line** into that same terminal. That is the command which starts the experiment.

The cell below only reviews files and prints commands. It does not start training, call an LLM, download data, or change your saved configuration. Missing data or an already-used output workspace produces a **STOP** instead of a launch command.

**Credentials are separate from the files in your project.** If you use a trusted shell environment file outside the project/repositories, make it readable only by you (`chmod 600 /absolute/path/to/credentials.env`). In the launching terminal, use:

```bash
set -a
source /absolute/path/to/credentials.env   # replace with YOUR existing trusted file
set +a
```

That file contains the required variable assignments, for example `OPENAI_API_KEY=...`; never paste real values into this notebook. Alternatively export keys through your managed secret system. Preview shows presence only; launch checks required keys again. Neither a notebook variable nor a file that has not been sourced configures the launching terminal.


In [ ]:
from IPython.display import Markdown, display

from tutorials.paper.launch_review import launch_review

RUN_EXAMPLE = "file-split"  # Choose ONE name from the table above.
if RUN_EXAMPLE not in {"initial", "longer", "less-trial", "budgets", "file-split"}:
    raise ValueError("Choose RUN_EXAMPLE from the table above")
config_name = (
    "tidmad-experiment" if RUN_EXAMPLE == "initial" else f"tidmad-{RUN_EXAMPLE}"
)
script_name = "tidmad" if RUN_EXAMPLE == "initial" else f"tidmad-{RUN_EXAMPLE}"
config_path = PROJECT / "experiments" / f"{config_name}.json"
script_path = PROJECT / "scripts" / f"run-{script_name}.sh"
selected = None
if config_path.is_file():
    selected = TidmadExperiment.model_validate_json(config_path.read_text())
if config_path.is_file() and script_path.is_file():
    display(Markdown(launch_review(config_path, script_path, task="tidmad")))
else:
    print(
        "Optional advanced example is not saved. The quick demo above is independent; enable the advanced save cell only if you want this additional example."
    )

### Hardware and execution limits for the search

The launcher accepts one RTX 5090 or H100 with working CUDA. Change `gpu` and budgets in the saved experiment, leaving memory headroom. Another NVIDIA GPU requires extending/qualifying the GPU check and validating the pinned CUDA environment with a small run; changing the name alone is not qualification. AMD and Intel GPUs are unsupported.

This fixed workflow runs under your local account without a separate security sandbox. Its time budgets are not total wall-time or API-cost caps. Existing blocking Health rules stay active; results are diagnostic. A fresh launch requires a new `workspace` and `run_name`, so rerunning a script against an existing output directory is refused.

## 7. Inspect the search before deciding whether to test

Wait for the section 6 terminal command to finish, or stop it deliberately. The command has trained and evaluated candidates using the task's **workflow validation** data. It has **not** run inference or scoring on final-test file 3.

Open the run record paths printed below. For each attempt, check `status`, its Trial/Formal role, its validation metric and its Health evidence together. A terminal exit or a saved checkpoint alone does not make a model eligible. Final-test selection requires a `success` attempt; choose it from validation evidence while the final-test result is still unknown.

- **You only wanted a configuration/run demo:** inspect the search records and stop here.
- **You selected `paper-pool`:** stop here; it is not the disjoint final-test example.
- **You selected `file-holdout` and have a successful model:** stop the search, select that model in section 8, then test it.
- **Every attempt failed:** inspect why and prepare a new search if needed. Do not weaken checks or look at final-test scores to decide your next model.


In [ ]:
if selected is not None:
    records = (
        sorted(selected.workspace.rglob("run_output_*.json"))
        if selected.workspace.exists()
        else []
    )
    print("Search result records:", *records, sep="\n")
    if not records:
        print("No completed records at", selected.workspace, "yet.")
print(
    "Final-test reports:",
    *sorted((PROJECT / "final-test").glob("*/result.json")),
    sep="\n",
)

## 8. Final test: freeze one selected model, then measure held-out performance

**This is not another training run and does not continue the search.** It uses the checkpoint you selected after section 7. It is available only for `file-holdout`.

| Action | Inputs | What it does | Output |
|---|---|---|---|
| Section 6: `run-tidmad-file-split.sh --launch` | Saved task and experiment; training/validation data | Calls the LLM, trains, validates and tunes | Search records/checkpoints in `runs/` |
| Here: `test-tidmad-file-split.sh ... --seal` | Your chosen checkpoint and its configs | Records hashes so the selection cannot silently change before testing; no inference | `selection.json` |
| Here: `test-tidmad-file-split.sh ... --evaluate` | That same sealed selection and held-out file 3 | Runs native inference and task scoring; no training, LLM, or feedback to search | Separate `result.json` |

`seal` is the record of your decision, not a second experiment. `evaluate` measures the selected model's held-out performance. We keep them separate so you can inspect the selected paths and hashes **before** seeing the test score. Sealing records selection; it does not stop a running search process for you.

Workflow validation can guide the agent and training/model selection. Final test must not. **First stop the search.** Use validation results and Health evidence to choose one successful completed attempt. The cell below lists the actual `run_output_*.json` files and attempt IDs in the selected experiment's workspace, so you do not need to guess model/checkpoint filenames.

The tutorial launcher explicitly retains original training checkpoints for this step. The helper resolves a chosen record to its checkpoint, model/loss configs and generated-library directory. It also checks that the model came from this experiment's task/split identity; a model trained on the paper pool is not eligible for the new file holdout.

Set `SELECTED_RUN_OUTPUT` to one printed path and `SELECTED_EXP_ID` to the corresponding attempt ID. Inspect the proposed candidate JSON. Only after confirming search has stopped, set `SEARCH_STOPPED=True` and `SAVE_FINAL_CANDIDATE=True` to save it as `final-test/candidate.json`. This saves configuration only: no test data is read and no inference is launched.

If every attempt reports `failed_mode_collapse` or another failure, stop here: there is no eligible final-test candidate. Tiny fractions and a one-epoch cap can exercise the software without learning a useful denoiser. Review validation evidence and revise a new search experiment; do not weaken the checks or use final-test data to choose those revisions.

In [ ]:
from agent.schemas.hyperparam_tuning import HyperparamTuningOutput

from tutorials.paper.tidmad.final_test import candidate_for_record

SELECTED_RUN_OUTPUT = None  # Path("/your/project/runs/.../run_output_iter_001.json")
SELECTED_EXP_ID = ""  # Copy the successful attempt ID you choose below.
SEARCH_STOPPED = False
SAVE_FINAL_CANDIDATE = False

if selected is not None and selected.protocol == "file-holdout":
    for result_path in sorted(selected.workspace.rglob("run_output_*.json")):
        result = HyperparamTuningOutput.model_validate_json(result_path.read_text())
        print("RUN RECORD:", result_path)
        for record in result.all_records:
            metric = record.metric_result.scalar if record.metric_result else None
            print(
                record.exp_id,
                record.status,
                "round",
                record.logical_round,
                "score",
                metric,
            )
            print("  Health:", record.health_gate_results)
    if SELECTED_RUN_OUTPUT is not None:
        candidate = candidate_for_record(
            selected,
            Path(SELECTED_RUN_OUTPUT),
            SELECTED_EXP_ID,
            search_completed=SEARCH_STOPPED,
        )
        print("PROPOSED CANDIDATE CONFIG:", candidate.model_dump_json(indent=2))
        if SAVE_FINAL_CANDIDATE:
            assert SEARCH_STOPPED, (
                "Stop search before selecting the final-test candidate"
            )
            candidate_path = PROJECT / "final-test/candidate.json"
            with candidate_path.open("x") as f:
                f.write(candidate.model_dump_json(indent=2))
            print("SAVED:", candidate_path)
else:
    print(
        "This final test requires the file-split experiment from sections 5–6, not paper-pool."
    )

Open [the saved candidate JSON](../final-test/candidate.json) to inspect the exact model paths. Then run these **two terminal commands**, substituting your project root:

```bash
/YOUR_PROJECT/scripts/test-tidmad-file-split.sh --candidate /YOUR_PROJECT/final-test/candidate.json --output /YOUR_PROJECT/final-test/test-001 --seal
/YOUR_PROJECT/scripts/test-tidmad-file-split.sh --candidate /YOUR_PROJECT/final-test/candidate.json --output /YOUR_PROJECT/final-test/test-001 --evaluate
```

`--seal` records selection hashes without test inference. `--evaluate` requires the same model/config/task, runs inference without training or LLM calls, then applies the selected task's metric and scoreability checks. Open `final-test/test-001/result.json` to see the separate result. It does **not** run the full workflow Health assessment and is not a paper-qualified score.

A started test cannot silently be retried in that directory. This is a procedural holdout, not an OS access barrier: files remain locally readable. Do not feed test scores back into model selection; if you adapt to them, you need a new holdout.

## 9. Which output should I open?

| Question | Open inside your project |
|---|---|
| Which task, experiment and command did the search use? | The `.tutorial.json` beside the selected run workspace |
| How did candidates perform on workflow validation? | `run_output_*.json` inside that run workspace; paths are listed in section 7 |
| Which model did I freeze for final testing? | `final-test/candidate.json` and `final-test/test-001/selection.json` |
| How did that selected model perform on held-out file 3? | `final-test/test-001/result.json`, only after section 8 completes |

The project root owns your editable inputs; each `runs/<name>/` is one search's output workspace. Console output stays in the launching terminal unless you redirect it to a project-local log. Final-test results are separate from search history and must not be fed back into model selection. A completed command does not establish a useful model or a paper-qualified result.
